In [8]:
import numpy as np
import pandas as pd
from tensorflow import keras
import ast
import joblib
from sklearn.metrics import mean_squared_error, r2_score
from keras import initializers
import tensorflow as tf

In [9]:
PREDICTORS = ["PwmD", "PwmE"]   
TARGET_INT = ["Theta"]  
TARGET = ["DeltaTheta"]
INPUT_SIZE = len(PREDICTORS)  
OUTPUT_SIZE = len(TARGET)   
TIME_STEPS = 3
TS = 0.07

In [10]:
df = pd.read_excel("resultados.xlsx")
df.head()

,model,Neurons,Ld,Lp,reg,R2_Train_1_Theta,MSE_Train_1_Theta,R2_Train_2_Theta,MSE_Train_2_Theta,R2_Val_Theta,MSE_Val_Theta,R2_Test_1_Theta,MSE_Test_1_Theta,R2_Test_2_Theta,MSE_Test_2_Theta,R2_Test_3_Theta,MSE_Test_3_Theta
0,model_arch16-8-4_r0.01_seed1133,"[16, 8, 4]",0.3,0.7,0.01,[0.8658295843195425],[0.030626628412947797],[0.7099773483105307],[0.06621312002931119],[-1.4099847289365268],[0.7887390659412833],[0.8312010342779467],[0.038776341956293196],[0.8450284891602868],[0.036753202686067694],[0.3696740878789889],[0.20721803225009539]
1,model_arch16-8-4_r0.01_seed592,"[16, 8, 4]",0.3,0.7,0.01,[0.8572725823768783],[0.032579906395263994],[0.38127270264851965],[0.14125746580929016],[-2.6684072272734904],[1.2005951968039013],[0.7407507974948662],[0.05955448651734243],[0.8214975447961],[0.042333825620731076],[0.6462999596262089],[0.11627798407717269]
2,model_arch16-8-4_r0.01_seed2344,"[16, 8, 4]",0.3,0.7,0.01,[0.8259711195705011],[0.03972498577279068],[0.7490657655987082],[0.05728910650629478],[-3.4654186093345096],[1.461440838472669],[0.8953507508714431],[0.024039928516861056],[0.866226797212467],[0.031725790175071995],[-1.2207490796878737],[0.7300655828437298]
3,model_arch16-8-4_r0.01_seed8964,"[16, 8, 4]",0.3,0.7,0.01,[0.8123752593621567],[0.042828466942194235],[0.7141652870147481],[0.0652569999246145],[0.7497979372603503],[0.08188605466763157],[0.8126836223522401],[0.04303014466121659],[0.8824490761603023],[0.027878497837473305],[-0.004056597486484126],[0.33008103966211666]
4,model_arch16-8-4_r0.01_seed7919,"[16, 8, 4]",0.3,0.7,0.01,[0.6480438737313285],[0.08033983827376022],[0.6417330656931395],[0.08179351297426551],[-3.1922745889736834],[1.3720463469225448],[0.8450971387655813],[0.035584141712838135],[0.8371145488004568],[0.03863008091042162],[0.041646911490108574],[0.3150561279221152]


In [11]:
Datasets = []
for i in range(4):
    Dataset = pd.read_excel(f"./../00-RotedData/Data.xlsx", f"D{i+1}")   
    Datasets.append(Dataset)

for i in range(4):   
    Dataset = pd.read_csv(f"./../00-Data/Data{i + 1}.csv")  
    Datasets.append(Dataset)
    
    
for i in range(len(Datasets)):
    Dataset = Datasets[i].copy()

    for var in TARGET_INT:
        Dataset[f"Delta{var}"] = (Dataset[var].shift(-1) - Dataset[var]) / TS
            
    Dataset = Dataset.dropna(subset=[f"Delta{var}" for var in TARGET_INT])

    Datasets[i] = Dataset

In [12]:
SCALER = joblib.load("./scalers/scaler.pkl")
OUT_SCALER = joblib.load("./scalers/out_scaler.pkl")

NormDatasets = []

for i in range(8):
    CurrentTestDataset = Datasets[i].copy()
    CurrentTestDataset[PREDICTORS] = SCALER.transform(CurrentTestDataset[PREDICTORS])
    CurrentTestDataset[TARGET] = OUT_SCALER.transform(CurrentTestDataset[TARGET])
    NormDatasets.append(CurrentTestDataset)

In [13]:
def BuildModel(architecture, r):

    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(TIME_STEPS, INPUT_SIZE)))

    for i, units in enumerate(architecture):

        if i < len(architecture) - 1:
            model.add(
                keras.layers.SimpleRNN(
                    units,
                    activation="tanh",
                    return_sequences=True,
                    kernel_regularizer=keras.regularizers.l2(r)
                )
            )
        else:
            model.add(
                keras.layers.SimpleRNN(
                    units,
                    activation="tanh",
                    kernel_regularizer=keras.regularizers.l2(r)
                )
            )

    model.add(keras.layers.Dense(OUTPUT_SIZE))

    return model

In [14]:
def LoadModelFromRow(row):

    # arquitetura
    arch = row["Neurons"]
    if isinstance(arch, str):
        arch = ast.literal_eval(arch)

    r = float(row["reg"])
    model_name = row["model"]

    weights_path = f"weights/{model_name}.weights.h5"

    # build
    model = BuildModel(arch, r)
    model.build((None, TIME_STEPS, INPUT_SIZE))

    # load pesos
    model.load_weights(weights_path)

    return model

In [19]:
def CreateSequences(input_data, target_data, timesteps):
    X_seq, Y_seq = [], []
    
    for i in range(timesteps, len(input_data)):
        X_seq.append(input_data.iloc[i-timesteps:i].values)
        Y_seq.append(target_data.iloc[i])
    return np.array(X_seq), np.array(Y_seq)


In [ ]:
R = tf.constant(0.0341, dtype=tf.float32)
L = tf.constant(0.0606, dtype=tf.float32)
dt = tf.constant(TS, dtype=tf.float32)

def CinematicModel(Wd, We, theta):

    dtheta_cin = (R / (2 * L)) * (Wd - We)
    # dx_cin = (R / 2) * tf.cos(theta) * (Wd + We)
    # dy_cin = (R / 2) * tf.sin(theta) * (Wd + We)

    # return [dtheta_cin, dx_cin, dy_cin]
    return [dtheta_cin]


def NumericalIntegration(dataset, dq):

    q = [None] * OUTPUT_SIZE

    init_vals = np.array([
        dataset[name].iloc[0] for name in TARGET_INT
    ])

    for j in range(OUTPUT_SIZE):
        q[j] = init_vals[j] + np.cumsum(dq[j] * TS)

    return q

def GetCin(dataset): 
    dq = CinematicModel(tf.convert_to_tensor(dataset["Wd"].values, dtype=tf.float32),
                        tf.convert_to_tensor(dataset["We"].values, dtype=tf.float32), 
                        tf.convert_to_tensor(dataset["Theta"].values, dtype=tf.float32))
    q = NumericalIntegration(dataset, dq)
    return np.vstack(q).T

In [28]:
# exemplo: melhor modelo por R2
best_idx = df["R2_Val_Theta"].idxmax()
row = df.loc[best_idx]

display(row)

model                model_arch16-8-4_r0.01_seed8964
Neurons                                   [16, 8, 4]
Ld                                               0.3
Lp                                               0.7
reg                                             0.01
R2_Train_1_Theta                [0.8123752593621567]
MSE_Train_1_Theta             [0.042828466942194235]
R2_Train_2_Theta                [0.7141652870147481]
MSE_Train_2_Theta               [0.0652569999246145]
R2_Val_Theta                    [0.7497979372603503]
MSE_Val_Theta                  [0.08188605466763157]
R2_Test_1_Theta                 [0.8126836223522401]
MSE_Test_1_Theta               [0.04303014466121659]
R2_Test_2_Theta                 [0.8824490761603023]
MSE_Test_2_Theta              [0.027878497837473305]
R2_Test_3_Theta              [-0.004056597486484126]
MSE_Test_3_Theta               [0.33008103966211666]
Name: 3, dtype: object

In [29]:
model = LoadModelFromRow(row)
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 simple_rnn_3 (SimpleRNN)    (None, 3, 16)             304       
                                                                 
 simple_rnn_4 (SimpleRNN)    (None, 3, 8)              200       
                                                                 
 simple_rnn_5 (SimpleRNN)    (None, 4)                 52        
                                                                 
 dense_1 (Dense)             (None, 1)                 5         
                                                                 
Total params: 561 (2.19 KB)
Trainable params: 561 (2.19 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [30]:
TITLES = ["Train_1", "Train_2", "Test_1", "Test_2", "Test_3", "Val", "LSG_1", "LSG_2"]

def EvalModel(model):
    n_targets = len(TARGET)

    # estrutura das métricas
    metrics = {
        name: {
            "R2_Train_1": [], "R2_Train_2": [],
            "R2_Val": [],
            "R2_Test_1": [], "R2_Test_2": [], "R2_Test_3": [],
            "R2_LSG_1": [], "R2_LSG_2": [],
            "MSE_Train_1": [], "MSE_Train_2": [],
            "MSE_Val": [],
            "MSE_Test_1": [], "MSE_Test_2": [], "MSE_Test_3": [],
            "MSE_LSG_1": [], "MSE_LSG_2": [],

        }
        for name in TARGET_INT
    }

    for i, dataset in enumerate(NormDatasets):

        x = dataset[PREDICTORS]
        y = dataset[TARGET]

        x, y = CreateSequences(x, y, TIME_STEPS)

        # predição da rede
        pred = model(tf.convert_to_tensor(x, dtype=tf.float32)).numpy()

        # desnormalização
        y_pred_diff = OUT_SCALER.inverse_transform(pred)
        y_diff = OUT_SCALER.inverse_transform(y)

        # arrays reconstruídos
        y_true = np.zeros_like(y_diff)
        y_pred = np.zeros_like(y_pred_diff)
        y_cin = GetCin(Datasets[i])
        
        n = y_true.shape[0]
        y_cin = y_cin[:n]

        # valores iniciais reais
        init_vals = np.array([
            Datasets[i][name].iloc[0] for name in TARGET_INT
        ])

        # reconstrução por integração cumulativa
        for j in range(n_targets):
            y_true[:, j] = init_vals[j] + np.cumsum(y_diff[:, j] * TS)
            y_pred[:, j] = init_vals[j] + np.cumsum(y_pred_diff[:, j] * TS)

        # cálculo das métricas
        for j, name in enumerate(TARGET_INT):

            r2 = r2_score(y_true[:, j], y_pred[:, j])
            mse = mean_squared_error(y_true[:, j], y_pred[:, j])

            metrics[name][f"R2_{TITLES[i]}"].append(r2)
            metrics[name][f"MSE_{TITLES[i]}"].append(mse)

            print(
                f"{name} | {TITLES[i]} -> "
                f"R² = {r2:.4f}, MSE = {mse:.4e}"
            )

    return metrics

In [31]:
metrics = EvalModel(model)
row

Theta | Train_1 -> R² = 0.7633, MSE = 5.4040e-02
Theta | Train_2 -> R² = 0.3936, MSE = 1.3844e-01
Theta | Test_1 -> R² = 0.3541, MSE = 1.4838e-01
Theta | Test_2 -> R² = 0.8517, MSE = 3.5165e-02
Theta | Test_3 -> R² = 0.6269, MSE = 1.2264e-01
Theta | Val -> R² = -4.5570, MSE = 1.8187e+00
Theta | LSG_1 -> R² = 0.5161, MSE = 2.0101e-01
Theta | LSG_2 -> R² = -1.5278, MSE = 8.8018e-01


model                model_arch16-8-4_r0.01_seed8964
Neurons                                   [16, 8, 4]
Ld                                               0.3
Lp                                               0.7
reg                                             0.01
R2_Train_1_Theta                [0.8123752593621567]
MSE_Train_1_Theta             [0.042828466942194235]
R2_Train_2_Theta                [0.7141652870147481]
MSE_Train_2_Theta               [0.0652569999246145]
R2_Val_Theta                    [0.7497979372603503]
MSE_Val_Theta                  [0.08188605466763157]
R2_Test_1_Theta                 [0.8126836223522401]
MSE_Test_1_Theta               [0.04303014466121659]
R2_Test_2_Theta                 [0.8824490761603023]
MSE_Test_2_Theta              [0.027878497837473305]
R2_Test_3_Theta              [-0.004056597486484126]
MSE_Test_3_Theta               [0.33008103966211666]
Name: 3, dtype: object